In [9]:
%%capture

import torch
from torch.nn.functional import cross_entropy
from transformers import DataCollatorForTokenClassification
import import_ipynb
from a_glance_at_dataset_and_tokenizer import xlmr_tokenizer
from create_model import device, XLMRobertaForTokenClassification
from tokenizing_text import panx_de_encoded, index2tag

In [3]:
xlmr_model_name = "xlm-roberta-base"
model_name = f"{xlmr_model_name}-finetuned-panx-de"

In [5]:
data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)
trained_model = XLMRobertaForTokenClassification.from_pretrained(
    model_name
).to(device)

In [6]:
def forward_pass_with_label(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    batch = data_collator(features)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():
        output = trained_model(input_ids, attention_mask)
        predicted_label = torch.argmax(output.logits, dim=-1).cpu().numpy()
    loss = cross_entropy(
        output.logits.view(-1, 7), labels.view(-1), reduction="none"
    )
    loss = loss.view(len(input_ids), -1).cpu().numpy()

    return {"loss": loss, "predicted_label": predicted_label}

In [15]:
valid_set = panx_de_encoded["validation"]
valid_set = valid_set.map(forward_pass_with_label, batched=True, batch_size=32)
df = valid_set.to_pandas()

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

In [16]:
index2tag[-100] = "IGN"
df["input_tokens"] = df["input_ids"].apply(
    lambda x: xlmr_tokenizer.convert_ids_to_tokens(x)
)
df["predicted_label"] = df["predicted_label"].apply(
    lambda x: [index2tag[i] for i in x]
)
df["labels"] = df["labels"].apply(
    lambda x: [index2tag[i] for i in x]
)
df["loss"] = df.apply(
    lambda x: x["loss"][:len(x["input_ids"])], axis=1
)
df["predicted_label"] = df.apply(
    lambda x: x["predicted_label"][:len(x["input_ids"])], axis=1
)

In [17]:
df.head(1)

,input_ids,attention_mask,labels,loss,predicted_label,input_tokens
0,"[0, 10699, 11, 15, 16104, 1388, 2]","[1, 1, 1, 1, 1, 1, 1]","[IGN, B-ORG, IGN, I-ORG, I-ORG, I-ORG, IGN]","[0.0, 0.010106107, 0.0, 0.018186016, 0.0132875...","[I-ORG, B-ORG, I-ORG, I-ORG, I-ORG, I-ORG, I-ORG]","[<s>, ▁Ham, a, ▁(, ▁Unternehmen, ▁), </s>]"
